In [ ]:
# --- Cell 1: Environment & Setup ---
import os, gc, re, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # ← 修正：改用 PyTorch 原生的 AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup # ← 修正：把這裡的 AdamW 拿掉
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler
from google.colab import files

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# --- Cell 2: Hyperparameters & Kaggle Download ---
import os
import kagglehub
from transformers import AutoTokenizer

# ==========================================
# 1. 設定 Kaggle 憑證 (替換成自己的 Username 與 Key)
# ==========================================
os.environ["KAGGLE_USERNAME"] = ""   # ← 修改這裡
os.environ["KAGGLE_KEY"]      = ""        # ← 修改這裡

# ==========================================
# 2. 自動下載競賽資料集
# ==========================================
print("⏳ 正在從 Kaggle 下載資料集...")
DATA_PATH = kagglehub.competition_download(
    "map-charting-student-math-misunderstandings"
)
print(f"✅ 資料集已下載至: {DATA_PATH}")
print(f"📂 包含檔案: {os.listdir(DATA_PATH)}")

# ==========================================
# 3. 建立超參數與動態路徑
# ==========================================
class CFG:
    MODEL_NAME = "microsoft/deberta-v3-base"
    MAX_LEN = 256
    BATCH_SIZE = 16
    EPOCHS = 3           # 單模型快速驗證，3 輪就夠
    LR = 2e-5

    # 將路徑動態指向 Kaggle 下載的資料夾
    TRAIN_PATH = os.path.join(DATA_PATH, "train.csv")
    TEST_PATH = os.path.join(DATA_PATH, "test.csv")

    # 兩階段模型的儲存路徑
    CAT_MODEL_PATH = "category_model_fold0.pt"
    MIS_MODEL_PATH = "miscon_model_fold0.pt"

tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME)

⏳ 正在從 Kaggle 下載資料集...
✅ 資料集已下載至: /root/.cache/kagglehub/competitions/map-charting-student-math-misunderstandings
📂 包含檔案: ['sample_submission.csv', 'train.csv', 'test.csv']


In [ ]:
# --- Cell 3: Data Preprocessing ---
def clean_text(s):
    if pd.isna(s): return ""
    s = str(s)
    s = re.sub(r'[,!?;\"\'\[\]\{\}]', ' ', s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_text(df):
    q = df["QuestionText"].fillna("").apply(clean_text)
    a = df["MC_Answer"].fillna("").apply(clean_text)
    e = df["StudentExplanation"].fillna("").apply(clean_text)
    df["text"] = q + " " + a + " " + e
    return df

# 讀取資料
train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
train = build_text(train)
test = build_text(test)

# --- 階段 1：Category 標籤 (6類) ---
cat_encoder = LabelEncoder()
train["cat_label"] = cat_encoder.fit_transform(train["Category"])
CAT_CLASSES = cat_encoder.classes_

# --- 階段 2：Misconception 標籤 (35類，僅取有迷思的資料) ---
train_mis = train[train["Misconception"].notna() & (train["Misconception"] != "NA")].copy()
mis_encoder = LabelEncoder()
train_mis["mis_label"] = mis_encoder.fit_transform(train_mis["Misconception"])
MIS_CLASSES = mis_encoder.classes_

print(f"Category 類別數: {len(CAT_CLASSES)}")
print(f"Misconception 類別數: {len(MIS_CLASSES)} (僅使用 {len(train_mis)} 筆資料訓練)")

Category 類別數: 6
Misconception 類別數: 35 (僅使用 9860 筆資料訓練)


In [ ]:
# --- Cell 4: Dataset & Training Engine ---
class MathDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=CFG.MAX_LEN, return_tensors="pt")
        item = {key: val.squeeze(0) for key, val in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def train_model(train_df, label_col, num_labels, save_path):
    # 簡單切出 80% 訓練, 20% 驗證 (單折)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_idx, val_idx = next(skf.split(train_df, train_df[label_col]))

    train_data = MathDataset(train_df.iloc[train_idx]["text"].values, train_df.iloc[train_idx][label_col].values)
    val_data = MathDataset(train_df.iloc[val_idx]["text"].values, train_df.iloc[val_idx][label_col].values)

    train_loader = DataLoader(train_data, batch_size=CFG.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=CFG.BATCH_SIZE*2, shuffle=False)

    # 強制設定 torch_dtype=torch.float32，確保優化器收到正確的梯度格式
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.MODEL_NAME,
        num_labels=num_labels,
        torch_dtype=torch.float32
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=CFG.LR, weight_decay=0.01)

    # 更新為 PyTorch 2.x 新版 AMP Scaler 語法
    scaler = torch.amp.GradScaler('cuda')

    best_loss = float('inf')
    for epoch in range(CFG.EPOCHS):
        model.train()
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()

            # 【關鍵修正 3】：更新為 PyTorch 2.x 新版 autocast 語法
            with torch.amp.autocast('cuda', dtype=torch.float16):
                outputs = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["labels"].to(device)
                )
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    outputs = model(
                        input_ids=batch["input_ids"].to(device),
                        attention_mask=batch["attention_mask"].to(device),
                        labels=batch["labels"].to(device)
                    )
                    val_loss += outputs.loss.item()

        val_loss /= len(val_loader)
        print(f"Epoch {epoch+1} - Val Loss: {val_loss:.4f}")
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), save_path)

    del model, optimizer; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- Cell 5: Train Category Model ---
print("🚀 開始訓練 Category 模型 (6 分類)...")
train_model(train, "cat_label", len(CAT_CLASSES), CFG.CAT_MODEL_PATH)

`torch_dtype` is deprecated! Use `dtype` instead!


🚀 開始訓練 Category 模型 (6 分類)...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Epoch 1:   0%|          | 0/1835 [00:00<?, ?it/s]

Epoch 1 - Val Loss: 0.7244


Epoch 2:   0%|          | 0/1835 [00:00<?, ?it/s]

Epoch 2 - Val Loss: 0.6618


Epoch 3:   0%|          | 0/1835 [00:00<?, ?it/s]

Epoch 3 - Val Loss: 0.5552


In [ ]:
# --- Cell 6: Train Misconception Model ---
print("🚀 開始訓練 Misconception 模型 (35 分類)...")
# 這裡只餵給模型「真的有迷思」的資料，讓他專心學錯誤邏輯
train_model(train_mis, "mis_label", len(MIS_CLASSES), CFG.MIS_MODEL_PATH)

🚀 開始訓練 Misconception 模型 (35 分類)...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Epoch 1:   0%|          | 0/493 [00:00<?, ?it/s]

Epoch 1 - Val Loss: 0.5095


Epoch 2:   0%|          | 0/493 [00:00<?, ?it/s]

Epoch 2 - Val Loss: 0.3312


Epoch 3:   0%|          | 0/493 [00:00<?, ?it/s]

Epoch 3 - Val Loss: 0.2768


In [ ]:
# --- Cell 7: Inference & Probability Multiplication ---
test_data = MathDataset(test["text"].values)
test_loader = DataLoader(test_data, batch_size=CFG.BATCH_SIZE*2, shuffle=False)

def get_predictions(model_path, num_labels):
    model = AutoModelForSequenceClassification.from_pretrained(CFG.MODEL_NAME, num_labels=num_labels).to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()

    all_probs = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            outputs = model(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
            probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    del model; gc.collect(); torch.cuda.empty_cache()
    return np.vstack(all_probs)

print("🔮 預測 Category 機率...")
cat_probs = get_predictions(CFG.CAT_MODEL_PATH, len(CAT_CLASSES))

print("🔮 預測 Misconception 機率...")
mis_probs = get_predictions(CFG.MIS_MODEL_PATH, len(MIS_CLASSES))

🔮 預測 Category 機率...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]

🔮 預測 Misconception 機率...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# --- Cell 8: Generate Final Submissions ---
TRUE_M_IDX = list(CAT_CLASSES).index("True_Misconception")
FALSE_M_IDX = list(CAT_CLASSES).index("False_Misconception")

final_rows = []
for b in range(len(test)):
    scores = {}

    # 1. 處理沒有迷思的類別 (e.g., True_Correct:NA)
    for j, c_name in enumerate(CAT_CLASSES):
        if c_name not in ["True_Misconception", "False_Misconception"]:
            scores[f"{c_name}:NA"] = cat_probs[b, j]

    # 2. 處理有迷思的類別 (機率相乘)
    for k, m_name in enumerate(MIS_CLASSES):
        scores[f"True_Misconception:{m_name}"] = cat_probs[b, TRUE_M_IDX] * mis_probs[b, k]
        scores[f"False_Misconception:{m_name}"] = cat_probs[b, FALSE_M_IDX] * mis_probs[b, k]

    # 3. 排序並取出機率最高的前 3 名
    top3 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]

    # 處理 ID 欄位 (可能是 id 也可能是 row_id)
    id_val = test.iloc[b]["row_id"] if "row_id" in test.columns else test.iloc[b].get("id", b)
    final_rows.append({
        "row_id": id_val,  # Kaggle 要求如果 test.csv 有 row_id
        "Category:Misconception": " ".join([x[0] for x in top3])
    })

sub_df = pd.DataFrame(final_rows)
# 強制將第一個欄位命名為競賽要求的名稱
sub_col_name = test.columns[0] if "row_id" not in test.columns and "id" not in test.columns else "row_id"
sub_df.rename(columns={"row_id": sub_col_name}, inplace=True, errors="ignore")

sub_df.to_csv("submission.csv", index=False)
print("✅ submission.csv 已經成功生成！")

✅ submission.csv 已經成功生成！


In [ ]:
# --- Cell 9: Download Submission ---
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>